# <span style="color:#2E86C1">CP3 · Comparativa de models de setmana 3 i decisió final abans d'obrir el test</span>

**NOTA**

Aquest notebook reprodueix els apartats 1–4 de [02_modelatge_baseline.ipynb](../notebooks/02_modelatge_baseline.ipynb) i s'**atura abans de l'apartat 5**. El conjunt de prova (gener–juny 2026) **no s'utilitza** en cap cel·la: totes les decisions es prenen només amb la validació de 2025.

## <span style="color:#16A085">Què aporta un model lineal regularitzat respecte de les referències?</span>

Les referències de setmana 2 són **regles fixes**:

- **Mitjana de grup:** dona el mateix valor tots els mesos → no veu l'estacionalitat ni la tendència.
- **Mateix mes de l'any anterior (`lag_12`):** recull l'estacionalitat, però només mira un únic dada i no reacciona si l'últim mes ha canviat o si hi ha campanya, descompte previst o pressupost diferent.

Un **model lineal** combina molts predictors a la vegada (retards, calendari, campanya, pressupost, descompte previst, estoc inicial, preu, canal, producte, regió) i aprén quin pes té cadascun.

La **regularització (ridge)** afegeix una penalització `alpha` als coeficients grans. Amb moltes variables i poques files (648 d'entrenament), un model lineal sense penalització pot aprendre soroll de l'entrenament (**sobreajust**). Ridge fa el model més estable: com més gran és `alpha`, més s'aplanen els coeficients. Triem `alpha` amb la validació.

## <span style="color:#16A085">1. Retards i particions</span>

Es calculen `lag_1` i `lag_12` dins de cada sèrie (canal + producte + regió) amb `shift`, de manera que només miren mesos anteriors. 2021 només serveix per construir `lag_12`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path('..') / 'dades'
sales = pd.read_csv(DATA / 'sales_history.csv')
sales['month'] = pd.to_datetime(sales['month'], format='%Y-%m')
KEY = ['channel', 'product_line', 'region']
sales = sales.sort_values(KEY + ['month']).reset_index(drop=True)
assert not sales.duplicated(['month'] + KEY).any()

for lag in [1, 12]:
    sales[f'lag_{lag}'] = sales.groupby(KEY)['sales_units'].shift(lag)
usable = sales.dropna(subset=['lag_1', 'lag_12']).copy()
train = usable[usable['month'] < '2025-01-01'].copy()
valid = usable[(usable['month'] >= '2025-01-01') & (usable['month'] < '2026-01-01')].copy()
# El test (2026) NO es crea ni s'utilitza en aquest notebook.

particions = pd.DataFrame([
    {'partició': 'entrenament', 'des de': train['month'].min().strftime('%Y-%m'),
     'fins a': train['month'].max().strftime('%Y-%m'), 'files': len(train)},
    {'partició': 'validació', 'des de': valid['month'].min().strftime('%Y-%m'),
     'fins a': valid['month'].max().strftime('%Y-%m'), 'files': len(valid)},
    {'partició': 'prova (reservada)', 'des de': '2026-01', 'fins a': '2026-06', 'files': 'no oberta'},
])
print(particions.to_string(index=False))

         partició  des de  fins a     files
      entrenament 2022-01 2024-12       648
        validació 2025-01 2025-12       216
prova (reservada) 2026-01 2026-06 no oberta


## <span style="color:#16A085">2. Mètriques, referències i model ridge</span>

Funcions del notebook 02 sense canvis:

- **MAE** = error mitjà en unitats. **WAPE** = error absolut total / vendes reals totals (%).
- Totes les prediccions es limiten al rang físic **0 – estoc inicial** (`cap`), igual per a tots els models → comparació justa.
- Ridge: imputació (mediana), codificació i escala **apreses només amb entrenament**.

In [2]:
def score(y, pred):
    y, pred = np.asarray(y, dtype=float), np.asarray(pred, dtype=float)
    return {'MAE_units': float(np.mean(np.abs(y-pred))),
            'WAPE_pct': float(100*np.abs(y-pred).sum()/np.abs(y).sum())}

group_mean = train.groupby(KEY)['sales_units'].mean().rename('prediction')
def mean_prediction(frame):
    merged = frame.merge(group_mean.reset_index(), on=KEY, how='left', validate='many_to_one')
    return merged['prediction'].fillna(train['sales_units'].mean()).to_numpy()

def cap(frame, prediction):
    return np.clip(prediction, 0, frame['stock_units'].to_numpy())

NUM = ['marketing_budget','planned_discount','stock_units','list_price',
       'days_in_month','lag_1','lag_12']
CAT = KEY + ['campaign']
def design(frame):
    x = frame[NUM].copy()
    month = frame['month'].dt.month
    x['month_sin'] = np.sin(2*np.pi*month/12)
    x['month_cos'] = np.cos(2*np.pi*month/12)
    x['time_index'] = (frame['month'].dt.year-2021)*12+month-1
    for product in ['urban','mountain','electric']:
        x[f'{product}_sin'] = x['month_sin']*(frame['product_line']==product)
        x[f'{product}_cos'] = x['month_cos']*(frame['product_line']==product)
    return pd.concat([x, pd.get_dummies(frame[CAT], dtype=float)], axis=1)

def fit_ridge(frame, alpha):
    x = design(frame)
    columns = x.columns
    medians = x.median()
    x = x.fillna(medians)
    center, scale = x.mean(), x.std().replace(0, 1)
    z = ((x-center)/scale).to_numpy(dtype=float)
    z = np.column_stack([np.ones(len(z)), z])
    penalty = np.eye(z.shape[1])*alpha
    penalty[0,0] = 0  # La constant no es penalitza.
    beta = np.linalg.solve(z.T@z+penalty, z.T@frame['sales_units'].to_numpy())
    return columns, medians, center, scale, beta

def predict_ridge(frame, model):
    columns, medians, center, scale, beta = model
    x = design(frame).reindex(columns=columns, fill_value=0).fillna(medians)
    z = ((x-center)/scale).to_numpy(dtype=float)
    return np.column_stack([np.ones(len(z)), z])@beta

print('Variables numèriques:', NUM)
print('Variables categòriques:', CAT)
print('Columnes finals del disseny:', len(design(train).columns))

Variables numèriques: ['marketing_budget', 'planned_discount', 'stock_units', 'list_price', 'days_in_month', 'lag_1', 'lag_12']
Variables categòriques: ['channel', 'product_line', 'region', 'campaign']
Columnes finals del disseny: 29


### <span style="color:#D68910">Variables usades i excloses</span>

| Tipus | Variables | Per què |
|---|---|---|
| **Admeses** (es coneixen abans del mes) | `channel`, `product_line`, `region`, `campaign`, `marketing_budget`, `planned_discount`, `stock_units` (estoc inicial), `list_price`, `days_in_month`, `lag_1`, `lag_12`, mes (sin/cos), `time_index` | Són planificades o del calendari, o són vendes de mesos ja observats |
| **Excloses** (es coneixen després) | `revenue`, `website_sessions`, `marketing_spend`, `avg_discount`, `closing_stock_units`, `stockout` | Usar-les del mateix mes seria fuga d'informació: el model "veuria el futur" |

## <span style="color:#16A085">3. Comparació en validació (2025)</span>

Mateixes 216 files de validació i mateixes mètriques per a tots els models. Ampliem la graella d'`alpha` del notebook base (1, 10, 100) amb 30, 300 i 1000 per veure on deixa de millorar.

In [3]:
ALPHAS = [1, 10, 30, 100, 300, 1000]
results = [
    {'model': 'group_mean', 'paràmetres': '-', **score(valid['sales_units'], cap(valid, mean_prediction(valid)))},
    {'model': 'seasonal_lag12', 'paràmetres': '-', **score(valid['sales_units'], cap(valid, valid['lag_12']))},
]
models = {}
for alpha in ALPHAS:
    name = f'ridge_{alpha:g}'
    models[name] = fit_ridge(train, alpha)
    results.append({'model': name, 'paràmetres': f'alpha={alpha:g}',
                    **score(valid['sales_units'], cap(valid, predict_ridge(valid, models[name])))})
comparison = pd.DataFrame(results).sort_values('MAE_units').reset_index(drop=True)
chosen = comparison.iloc[0]['model']
print(comparison.round(2).to_string(index=False))
print()
print('Model seleccionat només amb validació:', chosen)

         model paràmetres  MAE_units  WAPE_pct
     ridge_300  alpha=300      11.92     11.49
     ridge_100  alpha=100      12.05     11.61
      ridge_30   alpha=30      12.25     11.80
      ridge_10   alpha=10      12.37     11.92
       ridge_1    alpha=1      12.43     11.98
    ridge_1000 alpha=1000      13.32     12.83
seasonal_lag12          -      17.40     16.77
    group_mean          -      18.75     18.06

Model seleccionat només amb validació: ridge_300


### <span style="color:#D68910">Sobreajust: entrenament vs validació</span>

Si l'error d'entrenament fóra molt més baix que el de validació, el model estaria memoritzant.

In [4]:
rows = []
for name in ['ridge_1', chosen]:
    tr = score(train['sales_units'], cap(train, predict_ridge(train, models[name])))
    va = score(valid['sales_units'], cap(valid, predict_ridge(valid, models[name])))
    rows.append({'model': name, 'MAE_train': tr['MAE_units'], 'MAE_valid': va['MAE_units'],
                 'WAPE_train': tr['WAPE_pct'], 'WAPE_valid': va['WAPE_pct']})
print(pd.DataFrame(rows).round(2).to_string(index=False))

    model  MAE_train  MAE_valid  WAPE_train  WAPE_valid
  ridge_1      10.12      12.43       10.27       11.98
ridge_300      10.16      11.92       10.30       11.49


### <span style="color:#D68910">On falla més? (validació, model triat)</span>

In [5]:
v = valid[['month'] + KEY + ['campaign', 'stock_units', 'lag_1', 'lag_12', 'sales_units']].copy()
v['pred_ridge'] = cap(valid, predict_ridge(valid, models[chosen]))
v['pred_lag12'] = cap(valid, valid['lag_12'])
v['error_ridge'] = v['pred_ridge'] - v['sales_units']   # >0 sobreestima, <0 subestima
v['venut_tot_estoc'] = v['sales_units'] >= v['stock_units']

for col in ['channel', 'product_line']:
    t = v.groupby(col).apply(lambda g: pd.Series(score(g['sales_units'], g['pred_ridge'])), include_groups=False)
    print(t.round(2).to_string()); print()

print('Mitjana de l\'error (biaix) per canal:')
print(v.groupby('channel')['error_ridge'].mean().round(2).to_string())

           MAE_units  WAPE_pct
channel                       
ecommerce      14.06     11.95
store           9.78     10.88

              MAE_units  WAPE_pct
product_line                     
electric           8.42     12.19
mountain          11.75     11.59
urban             15.60     11.06

Mitjana de l'error (biaix) per canal:
channel
ecommerce   -7.30
store       -0.52


In [6]:
pd.set_option('display.width', 200)
top = v.reindex(v['error_ridge'].abs().sort_values(ascending=False).index).head(8)
top['month'] = top['month'].dt.strftime('%Y-%m')
print(top.round(1).to_string(index=False))

  month   channel product_line region  campaign  stock_units  lag_1  lag_12  sales_units  pred_ridge  pred_lag12  error_ridge  venut_tot_estoc
2025-06 ecommerce     mountain  metro      none          249  152.0   145.0          228       154.3       145.0        -73.7            False
2025-06 ecommerce        urban  coast      none          237  161.0   146.0          237       163.9       146.0        -73.1             True
2025-04 ecommerce        urban  metro      none          236  177.0   189.0          234       172.9       189.0        -61.1            False
2025-06 ecommerce        urban  metro      none          235  161.0   197.0          224       169.6       197.0        -54.4            False
2025-04 ecommerce     mountain  metro    spring          210  151.0   162.0          210       156.9       162.0        -53.1             True
2025-07 ecommerce     mountain  metro    summer          254  228.0   177.0          229       184.6       177.0        -44.4            False

## <span style="color:#16A085">4. Registre d'experiments</span>

Tots entrenats amb 2022–2024 (648 files) i avaluats amb 2025 (216 files).

| Model | Variables | Paràmetres | MAE (u.) | WAPE (%) | Observacions |
|---|---|---|---|---|---|
| Mitjana de grup | sèrie (canal+producte+regió) | – | 18,75 | 18,06 | No veu estacionalitat ni tendència |
| Mateix mes any anterior | `lag_12` | – | 17,40 | 16,77 | Millor referència; falla quan l'any canvia de nivell |
| Ridge | totes les admeses | alpha=1 | 12,43 | 11,98 | Ja millora un 29 % la millor referència |
| Ridge | totes les admeses | alpha=10 | 12,37 | 11,92 | |
| Ridge | totes les admeses | alpha=30 | 12,25 | 11,80 | |
| Ridge | totes les admeses | alpha=100 | 12,05 | 11,61 | Elecció per defecte del notebook base |
| **Ridge** | **totes les admeses** | **alpha=300** | **11,92** | **11,49** | **Millor en validació** |
| Ridge | totes les admeses | alpha=1000 | 13,32 | 12,83 | Massa penalització: el model s'aplana |

## <span style="color:#16A085">5. Dos casos d'error en validació</span>

**Cas 1 · Subestimació: e-commerce · urban · coast · juny 2025**
- Real: **237** u. · Ridge: **≈164** u. (error ≈ −73) · `lag_12`: 146 (error −91) · `lag_1` (maig): 161.
- Es va vendre **tot l'estoc** (237 = estoc inicial), així que la demanda real podria ser encara més alta: la xifra de vendes està "tallada" per l'estoc.
- El model es recolza en l'any anterior i en el mes previ, que van ser molt més baixos. Ni el model ni la referència capten el salt de l'e-commerce a l'estiu de 2025.
- **Per al negoci:** si s'hagueren comprat les unitats segons la previsió, hi hauria hagut ruptura d'estoc i vendes perdudes.

**Cas 2 · Sobreestimació: e-commerce · mountain · coast · maig 2025**
- Real: **110** u. · Ridge: **≈141** u. (error ≈ +31) · `lag_12`: 148 (error +38) · `lag_1` (abril): 146.
- Ací passa al revés: aquest mes es ven molt menys que l'any anterior i que el mes previ, sense campanya ni cap senyal planificat que ho avise, i el model no ho pot anticipar. És la sobreestimació més gran de la validació.
- **Per al negoci:** sobrarien unes 30 unitats d'estoc immobilitzat en una sèrie concreta.

**Patró general:** els errors més grans es concentren en l'**e-commerce** (MAE 14,06 vs 9,78 en botiga) i en la línia **urban** (MAE 15,60), i quasi sempre són **subestimacions en mesos amb molta demanda**: el biaix mitjà en e-commerce és −7,3 u. (el model es queda curt), mentre que en botiga és pràcticament zero (−0,5). A més, el 17 % de les files de validació venen tot l'estoc, i ací el valor real és un límit inferior de la demanda. Això és un límit que caldrà explicar a la setmana 5.

## <span style="color:#16A085">6. Decisió final (abans d'obrir el test)</span>

**Model triat: Ridge amb `alpha = 300`.**

- **Millora clara:** MAE 11,92 u. vs 17,40 u. de la millor referència (`lag_12`) → **−5,5 u. per sèrie i mes (−31 %)**. WAPE de 16,8 % a 11,5 %.
- **Compensa la complexitat?** Sí. El model continua sent lineal, ràpid i interpretable (es poden llegir els coeficients), i només afegeix un paràmetre (`alpha`). La millora respecte a la referència és molt més gran que les diferències entre valors d'`alpha` (només 0,5 u. entre alpha=1 i alpha=300).
- **Sense sobreajust greu:** MAE 10,16 en entrenament vs 11,92 en validació. Amb alpha=1 la distància és més gran (10,12 vs 12,43): la penalització fa el model més estable amb dades noves.
- **Per què 300 i no 100?** És el millor valor en validació; a partir de 1000 empitjora clarament, així que 300 està en la zona bona. La diferència amb 100 és xicoteta (0,13 u.), per tant la decisió és robusta.

**Què queda congelat per a la setmana 5:**
- Variables: les admeses de la taula anterior, sense afegir-ne ni llevar-ne.
- Entrenament: 2022–2024. Paràmetre: `alpha = 300`. Regla de límit: 0 – estoc inicial.
- El test de gener–juny 2026 s'avaluarà **una sola vegada** amb aquesta configuració, i no es canviarà res segons el resultat.

**Límits que ja sabem:**
- Subestima l'e-commerce en mesos de molta demanda.
- Quan es ven tot l'estoc, no coneixem la demanda real.
- Les dades són sintètiques: el resultat no és una previsió de mercat real.